# Samriddha Raj Satyal
## Batch 37B(AI)
## Coventry ID: 15940432
## Softwarica ID: 240488

In [50]:
from pyspark.sql import SparkSession,functions as F
from pyspark.sql.types import StructType, StructField,StringType,IntegerType,FloatType
from pyspark.sql.types import NumericType
from pyspark.sql.window import Window

In [51]:
spark=SparkSession.builder.appName('Supermarket Data Analysis').master('local[*]').getOrCreate()

# Part A: Data Loading and Inspection
## Q1. Load the supermarket sales dataset into a PySpark DataFrame.
## Q2. Display: - First 10 records - Schema of the dataset - Total number of rows and columns
## Q3. Check for: - Null values - Duplicate records
## Q4. Generate descriptive statistics for all numerical columns.

In [52]:
#Question 1
sales_df=spark.read.csv('supermarket_sales-ab9e0159-a52d-4f96-aa8c-19f4aa1530be.csv',inferSchema=True,header=True)


#Question 2
sales_df.show(10,truncate=False)
sales_df.printSchema()
print('Total Rows:',sales_df.count())
print('Total Columns:',len(sales_df.columns))


#Question 3
sales_df.select([F.sum(F.when(F.col(c).isNull(),1).otherwise(0)).alias(c)for c in sales_df.columns]).show()

total_count = sales_df.count()
unique_count = sales_df.dropDuplicates().count()
print(f"Total Duplicate Rows: {total_count - unique_count}")

#Question 4

numerical_cols = [
    field.name
    for field in sales_df.schema.fields
    if isinstance(field.dataType, NumericType)
]

sales_df.describe(*numerical_cols).show()

+--------------+----------+-----------+----------------+------------+--------+----------+-----------+--------------+----------+
|transaction_id|date      |customer_id|product_category|product_name|quantity|unit_price|total_sales|payment_method|city      |
+--------------+----------+-----------+----------------+------------+--------+----------+-----------+--------------+----------+
|T0001         |2025-02-05|C027       |Beverages       |Coffee      |5       |13.38     |66.9       |eWallet       |Biratnagar|
|T0002         |2025-02-25|C060       |Beverages       |Soft Drink  |1       |2.95      |2.95       |eWallet       |Biratnagar|
|T0003         |2025-04-25|C072       |Beverages       |Tea         |9       |21.85     |196.65     |Cash          |Pokhara   |
|T0004         |2025-03-28|C027       |Snacks          |Chocolate   |5       |9.04      |45.2       |Cash          |Bhaktapur |
|T0005         |2025-01-12|C118       |Beverages       |Juice       |6       |30.78     |184.68     |eWa

# Part B: EDA Using PySpark DataFrame Methods
## Q5. Calculate the total supermarket revenue.
## Q6. Find the top 5 highest-selling product categories based on total sales.
## Q7. Determine: - Average quantity sold - Average unit price - Average sales amount
## Q8. Find the city generating the highest revenue.
## Q9. Identify the payment method used most frequently.
## Q10. Calculate total sales for each product category.
## Q11. Find the top 10 transactions with the highest sales amount.
## Q12.
## Create a new column named sales_level: - High → sales > 100 - Medium → sales between 50 and 100 - Low → sales < 50

## Count the number of transactions in each sales level.

In [53]:
#Question 5
total_revenue=sales_df.select(F.sum('total_sales')).collect()[0][0]
print('Total Revenue is:',total_revenue)

#Question 6
sales_df.groupBy('product_category') \
    .agg(F.sum('total_sales').alias('Total_Sales')) \
    .orderBy(F.col('Total_Sales').desc()) \
    .show(5)


#Question 7
sales_df.select(
    F.avg('quantity').alias('Avg_Quantity_Sold'),
    F.avg('unit_price').alias('Avg_Unit_Price'),
    F.avg('total_sales').alias('Avg_Sales_Amount')
).show()

#Question 8
sales_df.groupBy('city') \
    .agg(F.sum('total_sales').alias('Total_Sales')) \
    .orderBy(F.col('Total_Sales').desc()) \
    .show(1)


#Question 9
sales_df.select(
    F.mode('payment_method').alias('Most frequent payment method')
).show()


#Question 10
sales_df.groupBy('product_category') \
    .agg(F.sum('total_sales').alias('Total_Sales')) \
    .show()


#Question 11
sales_df.orderBy(F.col('total_sales').desc()).select('transaction_id','total_sales').show(10)

#Question 12
sales_level_df = sales_df.withColumn(
    'sales_level',
    F.when(F.col('total_sales') > 100, 'High')
     .when((F.col('total_sales') >= 50) & (F.col('total_sales') <= 100), 'Medium')
     .otherwise('Low')
)

sales_level_df.groupBy('sales_level').count().show()

Total Revenue is: 72516.42
+----------------+------------------+
|product_category|       Total_Sales|
+----------------+------------------+
|          Snacks|15938.829999999998|
|           Dairy|15883.500000000004|
|          Bakery|14209.069999999994|
|       Beverages|14186.970000000005|
|         Produce|12298.050000000001|
+----------------+------------------+

+-----------------+------------------+----------------+
|Avg_Quantity_Sold|    Avg_Unit_Price|Avg_Sales_Amount|
+-----------------+------------------+----------------+
|            5.524|25.628980000000013|       145.03284|
+-----------------+------------------+----------------+

+---------+------------------+
|     city|       Total_Sales|
+---------+------------------+
|Bhaktapur|16650.069999999996|
+---------+------------------+
only showing top 1 row
+----------------------------+
|Most frequent payment method|
+----------------------------+
|                        Card|
+----------------------------+

+--------------

# Part C: EDA Using Spark SQL
## Q13.
## Create a temporary SQL view named:

## sales_view

## Q14.
## Using Spark SQL, write queries to answer the following:

## a)
## Total revenue generated by the supermarket.

## b)
## Revenue generated by each city.

## c)
## Revenue generated by each product category.

## d)
## Average quantity sold for each category.

## e)
## Top 5 products by total sales.

## f)
## Number of transactions for each payment method.

## g)
## Highest single transaction amount.

## h)
## Daily sales summary: - Date - Total Revenue - Number of Transactions

In [56]:
# Q13. Create a temporary SQL view
sales_df.createOrReplaceTempView("sales_view")

# Q14. Spark SQL Queries

# a) Total revenue generated by the supermarket
# FIXED: Changed SUM(Total) to SUM(total_sales)
spark.sql("SELECT SUM(total_sales) as Total_Revenue FROM sales_view").show()

# b) Revenue generated by each city
spark.sql("SELECT city, SUM(total_sales) as City_Revenue FROM sales_view GROUP BY city ORDER BY City_Revenue DESC").show()

# c) Revenue generated by each product category
spark.sql("SELECT product_category, SUM(total_sales) as Category_Revenue FROM sales_view GROUP BY product_category ORDER BY Category_Revenue DESC").show()

# d) Average quantity sold for each category
spark.sql("SELECT product_category, AVG(quantity) as Avg_Quantity FROM sales_view GROUP BY product_category").show()

# e) Top 5 products by total sales
spark.sql("SELECT product_name, SUM(total_sales) as Total_Sales FROM sales_view GROUP BY product_name ORDER BY Total_Sales DESC LIMIT 5").show()

# f) Number of transactions for each payment method
spark.sql("SELECT payment_method, COUNT(*) as Transaction_Count FROM sales_view GROUP BY payment_method ORDER BY Transaction_Count DESC").show()

# g) Highest single transaction amount
spark.sql("SELECT MAX(total_sales) as Highest_Transaction_Amount FROM sales_view").show()

# h) Daily sales summary
spark.sql("""
    SELECT date, SUM(total_sales) as Total_Revenue, COUNT(*) as Number_of_Transactions 
    FROM sales_view 
    GROUP BY date 
    ORDER BY Total_Revenue DESC
""").show(10)

+-------------+
|Total_Revenue|
+-------------+
|     72516.42|
+-------------+

+----------+------------------+
|      city|      City_Revenue|
+----------+------------------+
| Bhaktapur|16650.069999999996|
|  Lalitpur|16323.509999999998|
|   Pokhara|14540.320000000003|
| Kathmandu|          13034.83|
|Biratnagar|11967.690000000002|
+----------+------------------+

+----------------+------------------+
|product_category|  Category_Revenue|
+----------------+------------------+
|          Snacks|15938.829999999998|
|           Dairy|15883.500000000004|
|          Bakery|14209.069999999994|
|       Beverages|14186.970000000005|
|         Produce|12298.050000000001|
+----------------+------------------+

+----------------+------------------+
|product_category|      Avg_Quantity|
+----------------+------------------+
|          Bakery|5.6020408163265305|
|       Beverages| 5.245283018867925|
|           Dairy| 5.904255319148936|
|         Produce| 5.540816326530612|
|          Snacks|   

# Business Insights and Recommendations
## 1) The supermarket generated a Total Revenue of 72,516.42 across 500 transactions and Snacks was the category with highest revenue generation.
## 2) Bhaktapur leads all locations with the highest total revenue generated (16,650.07.
## 3) The average quantity sold was 5.524
## 4) The avg unit price was 25.628980000000013
## 5) The avg sales amount was 145.03284